<a href="https://colab.research.google.com/github/biopharma26/PracticeNotebooks/blob/main/03b_Build_RNA_Counts_Matrix.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03b - Build RNA Counts Matrix (input for Notebook 4)

**Objective:** Merge the per-sample GDC **STAR - Counts** files downloaded in `01_TCGA_Data_Download.ipynb` into a single `rna_counts_matrix.csv` (genes as rows, samples as columns) — the exact input Notebook 4 (`04_Differential_Expression.ipynb`) expects.

**Run this after Notebook 1** (you need the downloaded files in `data/rna_counts/`) **and before Notebook 4.**

**Why this step exists:** GDC's STAR-Counts files aren't a plain counts table — each file has a `gene-model` comment line, a header row, four STAR QC summary rows (`N_unmapped`, `N_multimapping`, `N_noFeature`, `N_ambiguous`), and multiple count columns (`unstranded`, `stranded_first`, `stranded_second`). This notebook parses that format correctly, picks the right column, and reshapes everything into one gene × sample matrix of raw integer counts.

In [1]:
import pandas as pd
import glob, os, re

RNA_DIR = "data/rna_counts"
files = [f for f in glob.glob(os.path.join(RNA_DIR, "*")) if not f.endswith((".csv",".json"))]
print(f"Found {len(files)} candidate count files in {RNA_DIR}")
files[:5]


Found 0 candidate count files in data/rna_counts


[]

## 1. Inspect One File
Sanity-check the format before batch-parsing. GDC STAR-Counts files typically look like:

```
# gene-model: GENCODE v36
gene_id            gene_name  gene_type       unstranded  stranded_first  stranded_second  tpm_unstranded  fpkm_unstranded  fpkm_uq_unstranded
N_unmapped         N_unmapped N_unmapped      1234567     0               0                0               0                0
N_multimapping     ...
N_noFeature        ...
N_ambiguous        ...
ENSG00000000003.15 TSPAN6     protein_coding  542         271             271              12.3            10.1             11.4
...
```

In [2]:
if files:
    sample_file = files[0]
    preview = pd.read_csv(sample_file, sep="\t", comment="#", nrows=10)
    print("Columns found:", list(preview.columns))
    preview
else:
    print("No files found — check that Notebook 1 downloaded files into data/rna_counts/")


No files found — check that Notebook 1 downloaded files into data/rna_counts/


## 2. Choose the Count Column
GDC RNA-seq at TCGA is typically **unstranded** library prep, so `unstranded` is almost always the correct raw-counts column for downstream DESeq2/PyDESeq2 analysis. If your library prep was stranded, switch `COUNT_COLUMN` to `stranded_first` or `stranded_second` (check the GDC case metadata / library strategy for your cohort if unsure).

In [ ]:
COUNT_COLUMN = "unstranded"
GENE_ID_COLUMN = "gene_id"          # Ensembl ID with version, e.g. ENSG00000000003.15
GENE_NAME_COLUMN = "gene_name"      # HGVS/Hugo symbol, e.g. TSPAN6

STAT_ROWS = {"N_unmapped", "N_multimapping", "N_noFeature", "N_ambiguous"}
print(f"Using count column: {COUNT_COLUMN}")


## 3. Map File -> Sample Barcode
The `rna_counts_manifest.csv` from Notebook 1 links each `file_id`/`file_name` to its `cases.submitter_id` (the TCGA sample barcode). We use that mapping so matrix columns are readable sample IDs rather than opaque file names.

In [ ]:
manifest_path = "rna_counts_manifest.csv"

if os.path.exists(manifest_path):
    manifest = pd.read_csv(manifest_path)
    print(manifest.shape)
    manifest.head()
else:
    manifest = pd.DataFrame()
    print("rna_counts_manifest.csv not found — matrix columns will fall back to file names instead of sample barcodes.")


In [ ]:
def resolve_sample_id(filepath, manifest):
    fname = os.path.basename(filepath)
    if manifest.empty:
        return fname
    # file names on disk were saved as "{file_id}_{file_name}" in Notebook 1
    match = manifest[manifest["file_name"].apply(lambda fn: isinstance(fn, str) and fn in fname)]
    if not match.empty:
        # prefer the submitter_id / case barcode column if present
        for col in ["cases.submitter_id", "submitter_id"]:
            if col in match.columns:
                val = match.iloc[0][col]
                if pd.notna(val):
                    return str(val)
        return str(match.iloc[0].get("file_id", fname))
    return fname

sample_ids_preview = [resolve_sample_id(f, manifest) for f in files[:5]]
sample_ids_preview


## 4. Parse and Merge All Count Files
Skips the `N_*` STAR QC rows, keeps raw integer counts, and merges everything into one wide matrix.

In [ ]:
from tqdm import tqdm

per_sample_series = {}
gene_name_lookup = None

for f in tqdm(files, desc="Parsing count files"):
    try:
        df = pd.read_csv(f, sep="\t", comment="#")
    except Exception as e:
        print(f"Skipping {f}: could not parse ({e})")
        continue

    if GENE_ID_COLUMN not in df.columns or COUNT_COLUMN not in df.columns:
        print(f"Skipping {f}: expected columns not found (got {list(df.columns)})")
        continue

    df = df[~df[GENE_ID_COLUMN].isin(STAT_ROWS)]
    df = df.set_index(GENE_ID_COLUMN)

    if gene_name_lookup is None and GENE_NAME_COLUMN in df.columns:
        gene_name_lookup = df[GENE_NAME_COLUMN]

    sample_id = resolve_sample_id(f, manifest)
    counts = pd.to_numeric(df[COUNT_COLUMN], errors="coerce").fillna(0).astype(int)

    if sample_id in per_sample_series:
        sample_id = f"{sample_id}__{os.path.basename(f)[:8]}"  # de-duplicate clashing barcodes

    per_sample_series[sample_id] = counts

print(f"Successfully parsed {len(per_sample_series)} samples")


In [ ]:
rna_counts_matrix = pd.DataFrame(per_sample_series)
rna_counts_matrix.index.name = "gene_id"

print("Matrix shape (genes x samples):", rna_counts_matrix.shape)
rna_counts_matrix.head()


## 5. Basic QC and Cleanup
- Drop genes with zero counts across every sample (uninformative for DESeq2/PyDESeq2)
- Report per-sample library size so you can spot failed/low-depth samples
- Optionally collapse duplicate Ensembl versions and attach gene symbols

In [ ]:
library_sizes = rna_counts_matrix.sum(axis=0).sort_values()
print("Smallest library sizes (check for failed samples):")
print(library_sizes.head(10))

low_depth_threshold = 1_000_000  # adjust as appropriate for your cohort
low_depth_samples = library_sizes[library_sizes < low_depth_threshold].index.tolist()
if low_depth_samples:
    print(f"\nWarning: {len(low_depth_samples)} sample(s) below {low_depth_threshold:,} total counts: {low_depth_samples}")


In [ ]:
genes_before = rna_counts_matrix.shape[0]
rna_counts_matrix = rna_counts_matrix.loc[rna_counts_matrix.sum(axis=1) > 0]
print(f"Removed {genes_before - rna_counts_matrix.shape[0]} all-zero genes; {rna_counts_matrix.shape[0]} genes remain")


## 6. Save Gene Symbol Lookup (optional but useful downstream)

In [ ]:
if gene_name_lookup is not None:
    gene_symbol_map = gene_name_lookup.reset_index()
    gene_symbol_map.columns = ["gene_id", "gene_name"]
    gene_symbol_map = gene_symbol_map.drop_duplicates(subset="gene_id")
    gene_symbol_map.to_csv("gene_symbol_map.csv", index=False)
    print("Saved gene_symbol_map.csv")
    gene_symbol_map.head()
else:
    print("No gene_name column available in source files — skipping symbol map.")


## 7. Save the Final Matrix
This is the exact file `04_Differential_Expression.ipynb` looks for: `rna_counts_matrix.csv`, genes as rows (index), samples as columns, raw integer counts.

In [ ]:
rna_counts_matrix.to_csv("rna_counts_matrix.csv")
print(f"Saved rna_counts_matrix.csv — shape {rna_counts_matrix.shape}")
print("\nColumns (sample IDs):")
print(list(rna_counts_matrix.columns)[:10], "...")


## 8. Cross-Check Against Stratification Groups
Confirms overlap between the matrix columns and the Mutant/WT sample lists from Notebook 3, so you know upfront how many samples Notebook 4 will actually be able to use.

In [ ]:
strat_path = "stratification_summary.csv"

if os.path.exists(strat_path):
    strat = pd.read_csv(strat_path)
    overlap = set(rna_counts_matrix.columns) & set(strat["Tumor_Sample_Barcode"])
    print(f"Samples in counts matrix: {rna_counts_matrix.shape[1]}")
    print(f"Samples in stratification file: {len(strat)}")
    print(f"Overlapping samples usable by Notebook 4: {len(overlap)}")
    if len(overlap) == 0:
        print("\nNo overlap found — check that sample barcode formats match "
              "(e.g. full TCGA barcode vs 12-character submitter ID). "
              "You may need to truncate/align IDs on both sides before running Notebook 4.")
else:
    print("stratification_summary.csv not found yet — run Notebook 3 first if you haven't.")


**Done.** `rna_counts_matrix.csv` is now ready — proceed to `04_Differential_Expression.ipynb`, which will load it directly instead of falling back to synthetic placeholder data.